# 📚 Build Mamba SSM from Scratch - Part 1: SSM Foundations

---

**Colab Ready**: This notebook runs on Google Colab with GPU support!

**Objectives**:
1. Understand the mathematical foundations of State Space Models (SSMs)
2. Derive the continuous-time to discrete-time conversion
3. Implement a basic SSM from scratch
4. Understand why SSMs matter for sequence modeling

---

## 1. What are State Space Models?

State Space Models (SSMs) are a class of models that represent sequences through:
- A **hidden state** $h(t)$ that captures the system's memory
- An **input** $x(t)$ that drives the system
- An **output** $y(t)$ that we want to predict

Think of it like a **continuous-time recurrence**: the future depends on the past through this hidden state.

## 2. Mathematical Foundations

### 2.1 Continuous-Time SSM

The continuous-time state space model is defined by:

$$
\begin{aligned}
h'(t) &= A h(t) + B x(t) \\
y(t) &= C h(t) + D x(t)
\end{aligned}
$$

Where:
- $h(t)$: Hidden state vector of dimension $N$
- $x(t)$: Input scalar (or vector)
- $y(t)$: Output scalar (or vector)
- $A \in \mathbb{R}^{N \times N}$: State transition matrix
- $B \in \mathbb{R}^{N \times 1}$: Input-to-state matrix
- $C \in \mathbb{R}^{1 \times N}$: State-to-output matrix
- $D \in \mathbb{R}$: Feed-through (often 0 for SSMs)

### 2.2 Intuition: What do A, B, C represent?
| Matrix | Role | Analogy |
|--------|------|--------|
| **A** | State dynamics | How the hidden state evolves over time |
| **B** | Input injection | How the input influences the state |
| **C** | State readout | How we extract output from state |

**Key insight**: The eigenvalues of $A$ determine the "memory" of the system:
- **Stable eigenvalues** ($|\lambda| < 1$): Information fades over time
- **Unit eigenvalues** ($|\lambda| = 1$): Information persists (perfect memory)
- **Unstable eigenvalues** ($|\lambda| > 1$): Information grows (avoid this!)

## 3. From Continuous to Discrete Time

### 3.1 The Discretization Problem

Neural networks operate on discrete sequences (tokens at positions $0, 1, 2, \dots$).
We need to convert the continuous-time ODE into a discrete-time recurrence.

### 3.2 Zero-Order Hold (ZOH) Discretization

The most common approach is **Zero-Order Hold (ZOH)** discretization:

We assume the input $x(t)$ is **constant** between discrete time steps:
$$
x(t) = x_k \quad \text{for} \quad t \in [k\Delta, (k+1)\Delta)
$$
where $\Delta$ is the **step size** (like token spacing).

### 3.3 Derivation

Starting from the continuous ODE:
$$
h'(t) = A h(t) + B x(t)
$$

This is a linear first-order ODE. The solution is:
$$
h(t) = e^{At}h(0) + \int_0^t e^{A(t-\tau)}B x(\tau) d\tau
$$

At time $t = \Delta$ (one step later):
$$
h(\Delta) = e^{A\Delta}h(0) + \int_0^{\Delta} e^{A(\Delta-\tau)}B x(\tau) d\tau
$$

With ZOH (input is constant $x_k$):
$$
h_{k+1} = e^{A\Delta}h_k + \left(\int_0^{\Delta} e^{A\tau} d\tau\right) B x_k
$$

Evaluating the integral:
$$
\int_0^{\Delta} e^{A\tau} d\tau = A^{-1}(e^{A\Delta} - I)\quad \text{(when } A \text{ is invertible)}
$$

### 3.4 The Final Discrete Form

Define:
$$
\begin{aligned}
\bar{A} &= e^{A\Delta} \\
\bar{B} &= (e^{A\Delta} - I) A^{-1} B \Delta
\end{aligned}
$$

The discrete-time SSM is:
$$
\begin{aligned}
h_k &= \bar{A} h_{k-1} + \bar{B} x_k \\
y_k &= C h_k
\end{aligned}
$$

**Note**: We drop $D$ (feed-through) as it's typically 0 in SSMs.

### 3.5 Special Case: When A is Diagonal

For efficiency, we often use **diagonal** $A$ matrices. Let $A = \text{diag}(\lambda_1, \dots, \lambda_N)$:

$$
\begin{aligned}
\bar{A}_{ii} &= e^{\lambda_i \Delta} \\
\bar{B}_i &= \frac{e^{\lambda_i \Delta} - 1}{\lambda_i} \cdot B_i
\end{aligned}
$$

This is much cheaper to compute! We'll use this in our implementation.

## 4. Setup and Imports

In [ ]:
# @title 🔧 Setup
# @markdown Run this cell to install dependencies and check GPU

!pip install torch numpy matplotlib -q

import torch
import numpy as np
import matplotlib.pyplot as plt

# Check for GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 5. Implementing the Basic SSM

In [ ]:
# @title 🧮 Basic SSM Implementation

class BasicSSM(torch.nn.Module):
    """
    Basic State Space Model - Sequential (Slow) Implementation
    
    This implements:
    h_k = A_bar * h_{k-1} + B_bar * x_k
    y_k = C * h_k
    """
    
    def __init__(self, d_model: int, d_state: int):
        super().__init__()
        self.d_model = d_model      # Output dimension
        self.d_state = d_state      # Hidden state dimension (N)
        
        # Initialize A, B, C matrices
        # A: d_state x d_state (we'll use diagonal form for efficiency)
        # B: d_state x 1 (input projection)
        # C: 1 x d_state (state readout)
        
        # First, let's initialize the continuous-time parameters
        # A will be diagonal - learnablelogarithms of the eigenvalues
        self.A_log = torch.nn.Parameter(torch.randn(d_state))
        
        # B and C are linear projections
        self.B_proj = torch.nn.Linear(d_model, d_state)
        self.C_proj = torch.nn.Linear(d_state, d_model)
        
        # Delta (step size) - can be learned or fixed
        self.delta = torch.nn.Parameter(torch.ones(d_state) * 0.1)
    
    def _discretize(self):
        """
        Apply ZOH discretization to get A_bar and B_bar
        
        A_log controls the eigenvalues (stability/memory)
        """
        # A_bar = exp(A * delta)
        # For diagonal A, this is simply exp(log(A) * delta) = exp(A_log * delta)
        A_bar = torch.exp(self.A_log * self.delta.unsqueeze(0))
        
        # B_bar = (exp(A*delta) - I) * A^(-1) * B * delta
        # = (A_bar - I) * A^(-1) * B * delta
        # For diagonal A: A_ii^(-1) = 1/A_ii
        # So: B_bar_i = (A_bar_i - 1) / A_log_i * delta * B_i
        # 
        # Special case: when A_log is close to 0, use L'Hôpital's rule
        # limit as A_log -> 0: (exp(a*d) - 1)/a * d -> d * d = delta^2
        
        A_inv = torch.where(
            torch.abs(self.A_log) > 1e-6,
            1.0 / self.A_log,
            self.delta  # fallback for small values
        )
        
        return A_bar, A_inv
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Input tensor of shape (batch, seq_len, d_model)
        Returns:
            y: Output tensor of shape (batch, seq_len, d_model)
        """
        batch, seq_len, d_model = x.shape
        
        # Discretize
        A_bar, A_inv = self._discretize()
        
        # Initialize hidden state
        h = torch.zeros(batch, self.d_state, device=x.device)
        
        outputs = []
        
        # Sequential (slow) computation - O(seq_len)
        for t in range(seq_len):
            # Get input at time t
            x_t = x[:, t, :]  # (batch, d_model)
            
            # Project input to state space
            B_t = self.B_proj(x_t)  # (batch, d_state)
            
            # B_bar = (A_bar - I) * A^(-1) * B * delta
            # Shape: (batch, d_state)
            B_bar_t = (A_bar.unsqueeze(0) - 1) * A_inv.unsqueeze(0) * B_t * self.delta.unsqueeze(0)
            
            # Update state: h = A_bar * h + B_bar * x
            h = A_bar.unsqueeze(0) * h + B_bar_t
            
            # Project to output
            y_t = self.C_proj(h)  # (batch, d_model)
            outputs.append(y_t)
        
        return torch.stack(outputs, dim=1)

# Test the basic SSM
print("Testing Basic SSM...")
ssm = BasicSSM(d_model=64, d_state=128).to(device)
x = torch.randn(2, 16, 64).to(device)  # (batch, seq_len, d_model)
y = ssm(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {y.shape}")

## 6. Understanding the SSM

Let's visualize what the SSM learns and how it processes sequences.

In [ ]:
# @title 🎨 Visualize SSM Behavior

# Let's understand what A_log controls
ssm = BasicSSM(d_model=4, d_state=8).to(device)

# Plot the eigenvalues (exp(A_log * delta))
A_bar, _ = ssm._discretize()
print("Eigenvalues (A_bar) - these control state dynamics:")
print(A_bar[0].detach().cpu().numpy())

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# 1. A_bar distribution
axes[0].hist(A_bar[0].detach().cpu().numpy(), bins=20, edgecolor='black')
axes[0].axvline(x=1.0, color='r', linestyle='--', label='Unit circle')
axes[0].set_xlabel('Eigenvalue magnitude')
axes[0].set_ylabel('Count')
axes[0].set_title('Eigenvalue Distribution')
axes[0].legend()

# 2. Test with a simple input sequence
x_test = torch.zeros(1, 20, 4).to(device)
x_test[0, 5, 0] = 1.0  # Single pulse at position 5
x_test[0, 10, 1] = 1.0  # Another pulse at position 10

with torch.no_grad():
    y_test = ssm(x_test)

# Plot response
for i in range(min(4, 4)):
    axes[1].plot(y_test[0, :, i].cpu().numpy(), label=f'Output {i}')
axes[1].axvline(x=5, color='k', linestyle=':', alpha=0.5)
axes[1].axvline(x=10, color='k', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Value')
axes[1].set_title('SSM Response to Pulses')
axes[1].legend()

# 3. Long-range dependency test
x_long = torch.zeros(1, 100, 4).to(device)
x_long[0, 0, 0] = 1.0  # First token
x_long[0, 99, 0] = 1.0  # Last token

with torch.no_grad():
    y_long = ssm(x_long)

axes[2].plot(y_long[0, :, 0].cpu().numpy())
axes[2].axvline(x=0, color='g', linestyle='--', alpha=0.5, label='First input')
axes[2].axvline(x=99, color='r', linestyle='--', alpha=0.5, label='Last input')
axes[2].set_xlabel('Position')
axes[2].set_ylabel('Value')
axes[2].set_title('Long-Range Dependency (100 tokens)')
axes[2].legend()

plt.tight_layout()
plt.savefig('ssm_behavior.png', dpi=150)
plt.show()

print("\n✅ Notice how the SSM can maintain information over long distances!")

## 7. Why is this interesting?

### 7.1 Comparison with RNNs

| Aspect | RNN | SSM |
|--------|-----|-----|
| State size | O(d_model) | O(d_state) |
| Computation | O(d_model²) | O(d_state²) or O(d_state) with diagonal |
| Parallelism | Sequential | Can be parallelized! |
| Gradient flow | Truncated BPTT | Full backprop through time |
| Long-range | Vanishing gradients | Better (controlled by A) |

### 7.2 The SSM Revolution

Recent work (S4, Mamba, etc.) has shown that:
1. **Diagonal + SSM = Powerful**: Can match or exceed transformers
2. **Fast inference**: O(1) per token (just matrix multiplication)
3. **Linear scaling**: No attention quadratic bottleneck
4. **Selective SSM**: Can learn what to remember/ignore (this is Mamba's key insight!)

## 8. Summary and Next Steps

In this notebook, we've learned:
1. ✅ The mathematical foundations of SSMs
2. ✅ How to discretize continuous-time SSMs using ZOH
3. ✅ How to implement a basic sequential SSM in PyTorch
4. ✅ Why SSMs are interesting for sequence modeling

**Key Takeaway**: The SSM transforms the sequence into a linear recurrence. The challenge is making it **selective** (learn what to ignore) and **parallelizable**.

---

**Next**: In [Part 2](./02_parallel_scan.ipynb), we'll learn about the **parallel scan algorithm** - the key insight that makes SSMs fast!

In [ ]:
# @title 📝 Exercise: Experiment with A_matrix

# @markdown Try modifying the A_log initialization to see different behaviors:
# @markdown - Large positive values: State grows quickly (unstable!)
# @markdown - Near zero: Long memory (close to unit circle)
# @markdown - Large negative: Short memory (quickly decays)

def create_ssm_with_memory_type(memory_type: str):
    """Create SSM with different memory characteristics"""
    ssm = BasicSSM(d_model=4, d_state=32).to(device)
    
    with torch.no_grad():
        if memory_type == 'long':
            # Near-zero A_log = unit circle = long memory
            ssm.A_log.fill_(0.01)
        elif memory_type == 'short':
            # Negative A_log = decay = short memory
            ssm.A_log.fill_(-2.0)
        elif memory_type == 'oscillatory':
            # Complex eigenvalues would oscillate, but we use real here
            ssm.A_log.fill_(0.1)
    
    return ssm

# Test different memory types
x_pulse = torch.zeros(1, 50, 4).to(device)
x_pulse[0, 10, 0] = 1.0

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for idx, mem_type in enumerate(['short', 'long', 'oscillatory']):
    ssm = create_ssm_with_memory_type(mem_type)
    with torch.no_grad():
        y = ssm(x_pulse)
    axes[idx].plot(y[0, :, 0].cpu().numpy())
    axes[idx].set_title(f'{mem_type.capitalize()} Memory')
    axes[idx].set_xlabel('Position')
    axes[idx].axvline(x=10, color='k', linestyle=':', alpha=0.5)

plt.tight_layout()
plt.show()